# Wartość klienta wg kanału pozyskania
Pytanie: czy różnice w revenue per user między kanałami to sygnał, czy szum?
Uruchamiaj komórka po komórce (Shift+Enter).

In [ ]:
# Konfiguracja

from google.cloud import bigquery

PROJECT_ID = "thelook-analytics-512312"   # Twój projekt GCP – to on "płaci" za zapytania
CUTOFF = "2026-10-01"                     # data odcięcia: tylko pełne miesiące
MAX_BYTES = 1_000_000_000                 # bezpiecznik: zapytanie > 1 GB zostanie odrzucone

client = bigquery.Client(project=PROJECT_ID)


def run_query(sql: str, params: list | None = None):
    """Szacuje koszt zapytania (dry run), a potem je wykonuje i zwraca DataFrame."""
    params = params or []

    # 1) Dry run – BigQuery liczy, ile danych przetworzy, ale niczego nie wykonuje (darmowe)
    dry = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            dry_run=True, use_query_cache=False, query_parameters=params
        ),
    )
    print(f"Zapytanie przetworzy: {dry.total_bytes_processed / 1e6:.2f} MB")

    # 2) Właściwe wykonanie z limitem kosztu
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=params, maximum_bytes_billed=MAX_BYTES
        ),
    )
    return job.to_dataframe()

In [ ]:
# Zapytanie: revenue per user z przedziałem ufności 95%

# @cutoff to parametr – wartość podajemy osobno, zamiast wklejać ją do tekstu SQL.
SQL_CHANNEL_CI = """
WITH user_revenue AS (
  SELECT
    u.id,
    u.traffic_source,
    COALESCE(SUM(oi.sale_price), 0) AS revenue
  FROM `bigquery-public-data.thelook_ecommerce.users` AS u
  LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` AS oi
    ON oi.user_id = u.id
    AND oi.status IN ('Complete', 'Shipped')
    AND oi.created_at < @cutoff
  WHERE u.created_at < @cutoff
  GROUP BY u.id, u.traffic_source
)
SELECT
  traffic_source,
  COUNT(*) AS users,
  AVG(revenue) AS revenue_per_user,
  STDDEV(revenue) / SQRT(COUNT(*)) AS std_error
FROM user_revenue
GROUP BY traffic_source
ORDER BY revenue_per_user DESC
"""

params = [bigquery.ScalarQueryParameter("cutoff", "TIMESTAMP", CUTOFF)]
df = run_query(SQL_CHANNEL_CI, params)

In [ ]:
# Przedziały ufności liczymy już w pandas

df["ci95_low"] = df["revenue_per_user"] - 1.96 * df["std_error"]
df["ci95_high"] = df["revenue_per_user"] + 1.96 * df["std_error"]
df.round(2)

In [ ]:
# Czy najlepszy i najgorszy kanał różnią się istotnie?

best, worst = df.iloc[0], df.iloc[-1]
overlap = best["ci95_low"] <= worst["ci95_high"]

print(f"Najwyżej: {best['traffic_source']} {best['revenue_per_user']:.2f} "
      f"[{best['ci95_low']:.2f}–{best['ci95_high']:.2f}]")
print(f"Najniżej: {worst['traffic_source']} {worst['revenue_per_user']:.2f} "
      f"[{worst['ci95_low']:.2f}–{worst['ci95_high']:.2f}]")
print("Przedziały zachodzą na siebie → brak dowodu na różnicę" if overlap
      else "Przedziały się nie pokrywają → różnica prawdopodobnie realna")